# V-Max BC pre-training on Colab (hanam/jeju, hard/easy pools)

Runs `algorithm=bc` training from `as-fast-as-anyone` (V-Max fork) on a Colab GPU instead of the local GTX 1660 Super (6GB VRAM / 15GB RAM).

**Data source**: the contest organizer's Drive folder, laid out as

```
Motion planning and prediction/train/
  hanam/<date>.tar.gz
  jeju/<date>.tar.gz
  livinglab/<date>.tar.gz     <- NOT used (the contest evaluates hanam/jeju)
```

Add that shared folder as a shortcut in your own Drive first (open the share -> "Add shortcut to Drive"), then fix `TRAIN_ROOT` below.

**What this notebook does**

1. `scripts/prepare_archives_91f.py` walks the per-date archives one at a time: extract -> convert to 91-step WOMD records (`make_91f`) -> score difficulty (`score_scenarios`) -> tar the result into a Drive cache -> delete the raw files. Peak local disk is one date, not the whole dataset, and a finished date is never converted twice (a killed session resumes by untarring the cache).
2. `scripts/split_hard_easy_pools.py --drop-frac 0.2 --hard-frac 0.4` throws away the bottom 20% (plain lane-keeping) per site and splits the rest into **4 pools**: `hanam_hard`, `hanam_easy`, `jeju_hard`, `jeju_easy`.
3. BC trains on all 4 as a weighted mixture, so the site ratio and the difficulty ratio are tuned independently.
4. Checkpoints go straight to Drive, and BC training resumes from the last one, so a disconnect mid-run costs one checkpoint interval.

Also upload the fixed 300-scenario **evaluation** set as a tar (so every model - local and Colab - is scored against the exact same set): locally, `tar -chf data/val_sample_shards_hanam.tar -C data/eval/val_sample_shards_hanam .` (~930MB, dereferenced so the symlinks survive), then upload it to `MyDrive/vmax_workdir/data/val_sample_shards_hanam.tar`.

Runtime > Change runtime type > pick a GPU. **Prefer an A100/L4 runtime if you have Colab Pro** - not for the GPU, for the vCPUs: the conversion step is pure CPU and a 2-vCPU T4 runtime converts roughly 6x slower.

In [ ]:
!nvidia-smi
!nproc && free -g && df -h /content

## 1. Mount Drive and check the layout

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Edit if the shared folder landed somewhere else.
TRAIN_ROOT = "/content/drive/MyDrive/Motion Planning and Prediction/train"
DRIVE_WORKDIR = "/content/drive/MyDrive/vmax_workdir"
SITES = "hanam,jeju"  # livinglab deliberately excluded
MAX_ARCHIVES_PER_SITE = 6  # dates per site, spread over its whole range; 0 for everything

os.environ["TRAIN_ROOT"] = TRAIN_ROOT
os.environ["DRIVE_WORKDIR"] = DRIVE_WORKDIR
os.environ["SITES"] = SITES
os.environ["MAX_ARCHIVES_PER_SITE"] = str(MAX_ARCHIVES_PER_SITE or 10**6)

assert os.path.isdir(TRAIN_ROOT), f"Missing {TRAIN_ROOT} - fix the path (did you Add shortcut to Drive?)."
for site in SITES.split(","):
    sdir = os.path.join(TRAIN_ROOT, site)
    assert os.path.isdir(sdir), f"Missing {sdir}"
    archives = sorted(f for f in os.listdir(sdir) if f.endswith((".tar.gz", ".tgz", ".tar")))
    print(f"{site}: {len(archives)} date archives, e.g. {archives[:3]}")

# Only needed for the checkpoint sweep in section 9 - training itself never reads it.
EVAL_TAR = f"{DRIVE_WORKDIR}/data/val_sample_shards_hanam.tar"
HAS_EVAL_TAR = os.path.exists(EVAL_TAR)
print("eval set:", "found" if HAS_EVAL_TAR else f"MISSING {EVAL_TAR} - sections 6/9 will skip it")

# 이후 모든 셀이 참조하는 run 이름 - 여기 한 곳에서만 고친다.
# Drive의 runs/에 실제로 있는 이름과 맞아야 한다 (섹션 6 뒤 점검 셀에서 확인).
# 계보: BC_RUN -> SAC_RUN -> FT_RUN -> RW_RUN (각 단계가 앞 단계의 체크포인트에서 출발)
BC_RUN = "colab_bc_run1"            # 섹션 7  BC 사전학습 (완료)
SAC_RUN = "colab_sac_easy1"         # 섹션 10 BC + 1차 SAC
FT_RUN = "colab_sac_finetune_v1"    # 섹션 11 + 2차 SAC (저 lr) <- 공식 채점 0.63281
RW_RUN = "colab_sac_collision_v4"   # 섹션 13 + 보상 재조정 (v1~v3 기록은 아래 참조)
for _k, _v in {"BC_RUN": BC_RUN, "SAC_RUN": SAC_RUN, "FT_RUN": FT_RUN, "RW_RUN": RW_RUN}.items():
    os.environ[_k] = _v
print("runs:", BC_RUN, SAC_RUN, FT_RUN, RW_RUN)


## 2. Clone the repo and set up the environment (uv, pinned by uv.lock)

In [ ]:
%cd /content
!rm -rf as-fast-as-anyone
!git clone https://github.com/gm2256/as-fast-as-anyone.git
%cd /content/as-fast-as-anyone/V-Max

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
# Installs its own Python 3.12 (per .python-version) regardless of Colab's system Python,
# and resolves the exact versions pinned in uv.lock (same env as the local machine).
!uv sync

## 3. Smoke test: one date archive per site

Do not skip this. It validates the Drive path, the archive contents and the record schema in a few minutes, and - more importantly - it prints **seconds per file**, which is the number to multiply out before committing to the full run in step 4.

In [ ]:
!uv run python scripts/prepare_archives_91f.py "$TRAIN_ROOT" /content/data/smoke_91f \
    --sites "$SITES" \
    --scores-dir /content/data/smoke_scores \
    --windows 100 \
    --max-archives-per-site 1
!find /content/data/smoke_91f -name '*.tfrecord' | wc -l
!du -sh /content/data/smoke_91f

In [ ]:
# Extrapolate before the real run: per-date file count and size x the dates you plan to convert.
import os

n_files = sum(len(fs) for _, _, fs in os.walk("/content/data/smoke_91f"))
size_gb = sum(
    os.path.getsize(os.path.join(d, f))
    for d, _, fs in os.walk("/content/data/smoke_91f") for f in fs
) / 1e9

sites = SITES.split(",")
n_all = sum(
    len([f for f in os.listdir(os.path.join(TRAIN_ROOT, s)) if f.endswith((".tar.gz", ".tgz", ".tar"))])
    for s in sites
)
n_planned = min(MAX_ARCHIVES_PER_SITE * len(sites), n_all) if MAX_ARCHIVES_PER_SITE else n_all
per_date_files, per_date_gb = n_files / len(sites), size_gb / len(sites)

print(f"smoke: {n_files} files, {size_gb:.1f} GB over {len(sites)} dates")
print(f"planned ({n_planned} of {n_all} dates): ~{per_date_files * n_planned:,.0f} files, "
      f"~{per_date_gb * n_planned:.1f} GB  <- must fit BOTH local disk and Drive")
print(f"everything ({n_all} dates): ~{per_date_files * n_all:,.0f} files, ~{per_date_gb * n_all:.1f} GB")

## 4. Full conversion + scoring, cached to Drive

`--cache-dir` makes this restartable: each finished date is tarred to `MyDrive/vmax_workdir/cache_91f/<site>/<date>.tar`, and a re-run untars it instead of reconverting. When the session dies, re-run sections 1-2 and then this cell unchanged.

Two things to watch:

- **`--windows 100`**: emits one 91-step window per source file instead of three, so the output is a third the size. Colab's local disk is ~112GB and the training pipeline needs the files locally, which the 3-window conversion does not fit; with one window the whole hanam+jeju set does. The dropped windows are other time slices of the *same* scene, so this costs far less than a third of the information. Remove the flag only if you have already capped the run enough to fit.
- **`--min-free-gb 15`**: stops the loop cleanly before an archive that would fill the disk (combined CSV still written, nothing half-done left behind) instead of dying on ENOSPC. Whatever converted so far is usable - just continue to step 5.
- **`--max-archives-per-site 6`**: 6 dates per site (of hanam's 31 and jeju's 14), spread evenly over each site's date range rather than taken from the front - a contiguous run of weeks is one season and one set of construction zones, which is what a policy overfits to. Start here: it reaches training in hours instead of a day, and raising the number later only converts the newly added dates.
- **Do not mix window settings**: dates already converted with 3 windows are skipped, not re-converted, so switching mid-way leaves some dates weighted 3x. To change it, wipe `/content/data/train_91f` and the Drive `cache_91f/` + `scores/` first.
- **Drive quota**: the cache holds the whole converted dataset (the estimate printed above).
- **Wall clock**: conversion is CPU-bound and Colab gives you 2-12 vCPUs. If the extrapolation says more hours than a session allows, that is fine (the cache resumes), but a capped run gets you to training sooner.

In [ ]:
!rm -rf /content/data/smoke_91f
!uv run python scripts/prepare_archives_91f.py "$TRAIN_ROOT" /content/data/train_91f \
    --sites "$SITES" \
    --scores-dir "$DRIVE_WORKDIR/scores" \
    --cache-dir "$DRIVE_WORKDIR/cache_91f" \
    --windows 100 \
    --min-free-gb 15 \
    --max-archives-per-site "$MAX_ARCHIVES_PER_SITE"
# Raise --max-archives-per-site (or drop it for everything) once a full run has gone
# through; already-converted dates come back from the Drive cache instead of reconverting.

## 5. Build the 4 training pools

`--drop-frac 0.2` discards the bottom 20% of each site outright (near-static lane keeping - the "직진 위주 단순 데이터 제거" step); `--hard-frac 0.4` then splits what remains into hard/easy **per site**, so `hanam_*` and `jeju_*` pools stay separately weightable.

`--holdout-frac 0.02` reserves 2% of each site as `<site>_val`, excluded from every training pool. That is what section 7 validates on: BC stops when *that* loss stops improving, which it cannot detect from the training loss alone (which keeps falling while the policy is memorising expert noise). The two per-site holdouts are merged into one `val` pool because `path_dataset_val` takes a single path.

In [ ]:
!rm -rf /content/data/shards/mixture_pools /content/data/shards/val
!uv run python scripts/split_hard_easy_pools.py \
    "$DRIVE_WORKDIR/scores/combined_scores.csv" \
    /content/data/train_91f \
    /content/data/shards/mixture_pools \
    --sites "$SITES" --drop-frac 0.2 --hard-frac 0.4 --holdout-frac 0.02

# One validation pool out of the per-site holdouts (path_dataset_val takes a single path).
!uv run python scripts/merge_pools.py /content/data/shards \
    val=/content/data/shards/mixture_pools/hanam_val,/content/data/shards/mixture_pools/jeju_val

## 6. Wire up checkpoints (Drive, persistent) and the fixed eval set

In [ ]:
import os
os.makedirs(f"{DRIVE_WORKDIR}/runs", exist_ok=True)
!rm -rf /content/as-fast-as-anyone/V-Max/runs
!ln -s "$DRIVE_WORKDIR/runs" /content/as-fast-as-anyone/V-Max/runs

if HAS_EVAL_TAR:
    !mkdir -p /content/data/eval/val_sample_shards_hanam
    !tar -xf "$EVAL_TAR" -C /content/data/eval/val_sample_shards_hanam
else:
    print("no eval tar - skipping (section 9 needs it)")

In [ ]:
# Drive에 실제로 남아 있는 run들 - 섹션 1에서 정한 이름과 맞는지 확인.
import os

R = "/content/as-fast-as-anyone/V-Max/runs"
for r in (sorted(os.listdir(R)) if os.path.isdir(R) else []):
    m = os.path.join(R, r, "model")
    n = len([f for f in os.listdir(m) if f.endswith(".pkl")]) if os.path.isdir(m) else 0
    print(f"{r}: {n} checkpoints")

print("\n섹션 1에서 정한 이름:", {k: os.environ.get(k) for k in ("BC_RUN", "SAC_RUN", "FT_RUN", "RW_RUN")})


## 7. Train

Weights below bias toward the evaluated site (hanam) and toward hard scenarios: hanam 0.6 / jeju 0.4, hard 0.6 / easy 0.4. Nothing is dropped inside a pool - the weight only sets how often each pool is drawn from - so this is safe to retune between runs.

`total_timesteps=5_000_000` is ~62.5k episodes at 80 steps, so with `--windows 100` (one scenario per source file) it is a few passes over the pool. Bump it (e.g. `20_000_000`, the framework default scale) once TensorBoard shows `train/imitation_loss` still trending down at 5M.

**Early stopping**: every `val_freq=100` iterations the current params are scored on the held-out `val` pool (same loss, same expert-driven unroll as training - only the scenarios differ). Each improvement rewrites `runs/<name_run>/model/model_best.pkl`; after `early_stop_patience=10` validations with no improvement the run stops and writes `early_stopped.txt`, which also makes a re-launch of the same `name_run` a no-op instead of training past the plateau. **Submit `model_best.pkl`, not `model_final.pkl`.** To only watch the curves without stopping, set `algorithm.early_stop_patience=0`; to disable validation entirely, `algorithm.val_freq=0`.

`early_stop_warmup_steps=1_000_000` is not optional. A tanh-output policy starts near zero and so do most expert actions, so the validation loss is *already low* at init and normally rises before it falls - without the warmup the run stops inside that transient and keeps the untrained network as its "best" (observed: stopped at 160k of 5M steps with best @ step 320). During the warmup the loss is logged but nothing is tracked or counted; the baseline is taken fresh at the first validation after it.

Watch these together in TensorBoard:

| curve | reading |
|---|---|
| `val/loss_vs_zero_policy` | **the one that says whether BC works at all.** 1.0 = no better than a policy that always outputs zero; expert actions cluster near zero, so the raw loss is small even untrained |
| `train/imitation_loss` vs `val/imitation_loss` | both falling = keep going; train falling while val flattens = the plateau early stopping is for |
| `val/imitation_loss_smoothed` | what patience actually judges - the raw value swings ~25% between validations from the params oscillating alone |

`batch_size=256` (up from the config's 64) is there to damp that oscillation; if the GPU runs out of memory, drop it back.

`policy.layer_sizes=[256,256]` overrides bc.yaml's `[256,64,32]` to match **sac.yaml**. This policy warm-starts SAC in section 10 and only same-shape arrays transfer, so training it at SAC's shape is what lets the hidden layers carry over and not just the encoder. It is also the shape `submission_hanam_run1/network.py` builds, so exported weights load there unchanged.

**Memory**: 4 pools means 4 live tf.data pipelines. If the session OOMs, drop `algorithm.buffer_size` to 10000 first, then `num_envs` to 2 - or merge back to 2 pools with `scripts/merge_pools.py`.

**If the session disconnects mid-run**: re-run sections 1-2, then 4 (restores from the Drive cache), 5 and 6, then this cell unchanged - `algorithm.resume=true` (default) picks up from `runs/<name_run>/model/train_state_latest.pkl` on Drive.

In [ ]:
import os

POOL_WEIGHTS = {"hanam_hard": 0.35, "hanam_easy": 0.25, "jeju_hard": 0.25, "jeju_easy": 0.15}
POOL_ROOT = "/content/data/shards/mixture_pools"

entries = []
for name, weight in POOL_WEIGHTS.items():
    pool_dir = os.path.join(POOL_ROOT, name)
    with open(os.path.join(pool_dir, "manifest.csv")) as fh:
        n = sum(1 for _ in fh) - 1  # minus header
    print(f"{name}: {n} shards, weight {weight}")
    entries.append(f"{{path: {pool_dir}/{name}.tfrecord@{n}, weight: {weight}}}")

MIXTURE = "[" + ", ".join(entries) + "]"
os.environ["MIXTURE"] = MIXTURE  # so the shell cell below sees it either way
print("\n" + MIXTURE)

with open("/content/data/shards/val/manifest.csv") as fh:
    n_val = sum(1 for _ in fh) - 1
VAL_PATH = f"/content/data/shards/val/val.tfrecord@{n_val}"
os.environ["VAL_PATH"] = VAL_PATH
print("validation:", VAL_PATH)

In [ ]:
# resume: true 이므로 같은 이름으로 다시 실행하면 새 학습이 아니라 이어학습이 된다.
%cd /content/as-fast-as-anyone/V-Max
!uv run python vmax/scripts/training/train.py \
  algorithm=bc network/encoder=lq \
  algorithm.network.policy.layer_sizes=[256,256] \
  total_timesteps=5_000_000 num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=20000 \
  waymo_dataset=true \
  "mixture_datasets=$MIXTURE" \
  "path_dataset_val=$VAL_PATH" num_scenario_per_val=64 \
  algorithm.val_freq=100 algorithm.early_stop_patience=10 algorithm.early_stop_warmup_steps=1_000_000 \
  algorithm.batch_size=256 \
  name_run=$BC_RUN log_freq=50 save_freq=1500

## 8. Watch training in TensorBoard (optional, run in a separate cell while training runs)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/as-fast-as-anyone/V-Max/runs

## 9. After training: sweep checkpoints on held-out scenarios

This is the number that decides things, not the loss: each checkpoint drives the scenarios itself and is scored on offroad / collision / progress, the way the contest scores. `rideflux_aggregate_score` is the column closest to the contest's own.

Without the uploaded 300-scenario tar this falls back to the `val` pool from section 5 - the same scenarios early stopping selected `model_best.pkl` on, so its score here is mildly optimistic relative to the others. Scores from different sets are never comparable with each other, so keep every run on one of them.

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
import os

# The fixed 300-scenario set if it was uploaded (comparable with the local runs),
# otherwise the held-out pool from section 5.
EVAL_PATH = (
    "/content/data/eval/val_sample_shards_hanam/val_sample_shards_hanam.tfrecord@300"
    if HAS_EVAL_TAR
    else VAL_PATH
)
os.environ["EVAL_PATH"] = EVAL_PATH
print("scoring against:", EVAL_PATH)

# Sweeps every model_*.pkl, so it also scores model_best.pkl (the early-stopping pick).
!uv run python scripts/evaluate_checkpoints.py \
  --name_run "$BC_RUN" \
  --path_dataset "$EVAL_PATH" \
  --waymo_dataset true --batch_size 4

## 10. RL fine-tuning: SAC warm-started from the BC policy

BC only ever sees expert-driven states, so it never learns to recover from its own mistakes - that is what RL is for, and starting RL from the BC weights skips the phase where a random policy crashes its way to a first reward.

**The transfer is partial, by construction.** SAC's policy emits the parameters of a distribution over actions (size 4: mean and std) where BC's emits the action itself (size 2), and SAC additionally has value networks BC never had. `pretrained_params_path` therefore grafts every array whose path *and* shape match and leaves the rest at its fresh init - with section 7's `[256,256]` and the same encoder, that is everything except the head and the value networks.

The run prints what actually transferred - `grafted N arrays, kept M at fresh init`. If N is 0 the configs do not line up; check the BC run used the same `network/encoder` and `policy.layer_sizes`. A BC run trained at bc.yaml's default `[256,64,32]` still transfers its encoder here (the bulk of the parameters), just not the hidden layers.

Point it at `model_best.pkl` (the validation-selected checkpoint), not `model_final.pkl` - on the first BC run those scored 0.585 and 0.414 on the held-out pool.

`learning_start=2000` matches the earlier local SAC runs: taking thousands of updates off a near-empty buffer is the fastest way to undo the pre-training. `total_timesteps=1_000_000` is a place to stop and compare against the BC score, not a target - raise it and re-run the same `name_run` to continue.

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
import os
BC_WEIGHTS = f"runs/{BC_RUN}/model/model_best.pkl"
os.environ["BC_WEIGHTS"] = BC_WEIGHTS
assert os.path.exists(BC_WEIGHTS), f"missing {BC_WEIGHTS}"
print("warm start from:", BC_WEIGHTS)

In [ ]:
!uv run python vmax/scripts/training/train.py \
  algorithm=sac network/encoder=lq \
  algorithm.network.policy.layer_sizes=[256,256] \
  "algorithm.pretrained_params_path=$BC_WEIGHTS" \
  total_timesteps=1_000_000 num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=50000 algorithm.learning_start=2000 \
  waymo_dataset=true \
  "mixture_datasets=$MIXTURE" \
  name_run=$SAC_RUN log_freq=50 save_freq=500

### Score the SAC checkpoints on the same held-out pool

Same command as section 9 with the SAC run name, so the numbers sit on the same scale as the BC ones - that comparison is the only thing that says whether RL helped.

In [ ]:
!uv run python scripts/evaluate_checkpoints.py \
  --name_run "$SAC_RUN" \
  --path_dataset "$EVAL_PATH" \
  --waymo_dataset true --batch_size 4

## 11. Second pass: fine-tune the SAC policy at a lower learning rate

Section 10's run starts from BC and explores hard. This pass takes its best-scoring checkpoint and settles it down: a smaller learning rate, less entropy pressure, and a hard-weighted mixture (`POOL_WEIGHTS` below - hard 0.8 against section 7's 0.6).

`pretrained_params_path` pointed at another SAC run's checkpoint takes the params **whole** - policy, both value networks, their targets - because the struct matches. The optimizer state is not carried over, so Adam's moments restart; that is what makes a learning-rate change take effect cleanly rather than fighting stale moments.

The knobs this repo's SAC actually exposes are `algorithm.learning_rate` (one value, used to build both the actor and the critic optimizer) and `algorithm.alpha` (a **fixed** entropy coefficient - alpha is not auto-tuned here, so there is no target entropy to scale). Anything else under `algorithm.` fails at config composition, so `actor_lr` / `critic_lr` / `target_entropy_scale` are not available.

`learning_start` is the number of **random-action** prefill steps. Keep it small but nonzero: a warm-started policy gains nothing from a random-action buffer, but at 0 the first updates resample the same `num_envs` transitions over and over.

Checkpoints land in `runs/<name_run>/model/model_<step>.pkl` - the Drive symlink from section 6 - which is where the checkpoint path below has to point.

In [ ]:
import os

POOL_WEIGHTS = {"hanam_hard": 0.5, "hanam_easy": 0.1, "jeju_hard": 0.3, "jeju_easy": 0.1}
POOL_ROOT = "/content/data/shards/mixture_pools"

entries = []
for name, weight in POOL_WEIGHTS.items():
    pool_dir = os.path.join(POOL_ROOT, name)
    with open(os.path.join(pool_dir, "manifest.csv")) as fh:
        n = sum(1 for _ in fh) - 1  # minus header
    print(f"{name}: {n} shards, weight {weight}")
    entries.append(f"{{path: {pool_dir}/{name}.tfrecord@{n}, weight: {weight}}}")

MIXTURE = "[" + ", ".join(entries) + "]"
os.environ["MIXTURE"] = MIXTURE  # so the shell cell below sees it either way
print("\n" + MIXTURE)

with open("/content/data/shards/val/manifest.csv") as fh:
    n_val = sum(1 for _ in fh) - 1
VAL_PATH = f"/content/data/shards/val/val.tfrecord@{n_val}"
os.environ["VAL_PATH"] = VAL_PATH
print("validation:", VAL_PATH)

# Section 9's held-out set, so section 11 can be run without replaying section 9.
EVAL_PATH = (
    "/content/data/eval/val_sample_shards_hanam/val_sample_shards_hanam.tfrecord@300"
    if HAS_EVAL_TAR
    else VAL_PATH
)
os.environ["EVAL_PATH"] = EVAL_PATH
print("scoring against:", EVAL_PATH)

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
import os
import shutil

# 1. 파인튜닝 기준 체크포인트 지정.
#    같은 name_run으로 다시 돌리면 model_final.pkl이 덮어써지므로 사본을 남긴다.
MODEL_DIR = f"runs/{os.environ['SAC_RUN']}/model"
FT_BASE = f"{MODEL_DIR}/model_1000000_best.pkl"

if not os.path.exists(FT_BASE):
    src_ckpt = f"{MODEL_DIR}/model_final.pkl"
    if not os.path.exists(src_ckpt):
        have = sorted(os.listdir(MODEL_DIR)) if os.path.isdir(MODEL_DIR) else []
        raise FileNotFoundError(f"missing {src_ckpt}; run dir holds: {have}")
    shutil.copy2(src_ckpt, FT_BASE)
    print("preserved", src_ckpt, "->", FT_BASE)

os.environ["FT_BASE"] = FT_BASE
print("fine-tune from:", FT_BASE)


In [ ]:
# 2. 파인튜닝 실행
!uv run python vmax/scripts/training/train.py \
  algorithm=sac network/encoder=lq \
  algorithm.network.policy.layer_sizes=[256,256] \
  "algorithm.pretrained_params_path=$FT_BASE" \
  algorithm.learning_rate=6e-5 \
  algorithm.alpha=0.1 \
  total_timesteps=150_000 \
  num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=50000 algorithm.learning_start=200 \
  waymo_dataset=true \
  "mixture_datasets=$MIXTURE" \
  name_run=$FT_RUN log_freq=20 save_freq=100

### Score the fine-tuned checkpoints on the same held-out pool

`$EVAL_PATH` is whatever section 9 picked, so these numbers sit on the same scale as the BC and the section 10 SAC ones - without that, the fine-tune cannot be told apart from the run it started from.

In [ ]:
!uv run python scripts/evaluate_checkpoints.py \
  --name_run "$FT_RUN" \
  --path_dataset "$EVAL_PATH" \
  --waymo_dataset true --batch_size 4

## 12. 대회 공식 채점기로 평가

섹션 9/11의 `evaluate_checkpoints.py`는 V-Max 파이프라인이고, 실제 순위는
`dxchallenge_planning_eval/evaluate.py`가 매긴다. 둘의 점수가 비슷해야 제출물이
정상이라는 뜻이므로, **같은 평가셋**(`$EVAL_PATH`)으로 양쪽을 돌려 비교한다.

`error` 컬럼이 0이 아니면 제출물이 JAX-traceable하지 않거나 예외를 던진 것이고,
그 시나리오들은 0점 처리된다.


In [ ]:
# 평가기는 V-Max와 별개 프로젝트다 (순수 waymax, vmax 패키지 불필요).
# uv가 별도 .venv를 만들지만 휠은 캐시에서 하드링크되므로 디스크 추가분은 작다.
import os

os.chdir("/content/as-fast-as-anyone/dxchallenge_planning_eval")
!uv sync
print("eval venv:", os.path.isdir(".venv"))


In [ ]:
# V-Max 체크포인트 -> 대회 제출 형식.
# submission_hanam_run1을 틀로 쓰고 weights.pkl만 갈아끼운다. actor.py / network.py /
# feature_extractor.py는 hanam_run1 설정 기준이므로, 다른 run을 포장할 때는
# OBSERVATION_CONFIG와 policy layer_sizes가 그 run의 .hydra/config.yaml과 같아야 한다.
# 어긋나면 에러 없이 엉뚱한 점수가 나온다 - 아래 두 grep으로 반드시 대조할 것.
import os
import shutil

os.chdir("/content/as-fast-as-anyone/V-Max")

SUB_RUN = os.environ["FT_RUN"]   # 포장할 run
SUB_CKPT = "model_96320.pkl"     # 그 run에서 스윕으로 고른 체크포인트
SUB_DIR = "/content/as-fast-as-anyone/dxchallenge_planning_eval/submission_best"

ckpt = f"runs/{SUB_RUN}/model/{SUB_CKPT}"
assert os.path.isfile(ckpt), f"없음: {ckpt}"
shutil.copytree(
    "/content/as-fast-as-anyone/dxchallenge_planning_eval/submission_hanam_run1",
    SUB_DIR, dirs_exist_ok=True,
)
!uv run python scripts/export_policy_weights.py "{ckpt}" "{SUB_DIR}/weights.pkl"

print("\n--- run 설정 ---")
!grep -A12 "observation_config:" runs/{SUB_RUN}/.hydra/config.yaml
print("\n--- 제출물 설정 ---")
!grep -A20 "OBSERVATION_CONFIG" {SUB_DIR}/actor.py


In [ ]:
# 이 평가기는 waymax 기본값(roadgraph 30000점)으로 데이터를 읽는데, 우리 변환본은
# make_91f.py의 ROADGRAPH_KEEP_POINTS=10000이다. 자체 샤드로 채점하려면 맞춰야 한다.
# (주최측 원본 validation tfrecord로 돌릴 때는 이 셀을 건너뛴다.)
# 섹션 2가 repo를 새로 clone하므로 세션마다 다시 실행해야 한다.
import os

os.chdir("/content/as-fast-as-anyone/dxchallenge_planning_eval")
!sed -i 's/DatasetConfig(path=path, repeat=1, shuffle_seed=None)/DatasetConfig(path=path, repeat=1, shuffle_seed=None, max_num_rg_points=10000)/' evaluate.py
assert "max_num_rg_points=10000" in open("evaluate.py").read(), "패치 실패 - evaluate.py의 DatasetConfig 줄을 직접 확인"
print("패치 완료")


In [ ]:
import os

os.chdir("/content/as-fast-as-anyone/dxchallenge_planning_eval")

# 섹션 9/11과 같은 평가셋을 써야 V-Max 스윕 점수와 나란히 놓을 수 있다.
print("평가셋:", os.environ["EVAL_PATH"])

# BATCH_SIZE는 actor.py의 모듈 상수(기본 64)를 그대로 쓴다. T4에서 OOM이 나면
# 그 값을 임시로 낮추되, 제출 전에는 반드시 64로 되돌릴 것 - 낮은 값으로 제출하면
# 5만 시나리오 / 30분 제한에 걸려 0점이 된다.
!CUDA_VISIBLE_DEVICES=0 uv run evaluate.py \
    --path_dataset "$EVAL_PATH" \
    --submission submission_best \
    --time_limit 0


## 13. 보상 재조정: 충돌 억제

`colab_sac_finetune_v1/model_96320.pkl`은 공식 채점기에서 **0.63281**인데, 품질 항
`(7×0.896 + 3×0.647)/10 = 0.821`을 벌어놓고 **게이트에서 23%를 잃는다**
(충돌 20.6%, 이탈 8.2%). 부딪히지만 않으면 0.821이므로, 남은 점수의 대부분이
여기 있다.

### 왜 정책이 충돌을 감수하는가

기본 보상은 `expert_progression: 10.0` 대 `overlap: -1.0`이고,
`termination_keys`에 `overlap`이 있어 충돌 시 에피소드가 **종료**된다. 그래서
충돌의 실제 비용은 `-1 + (1 - progress) × 10`이다:

| 충돌 시점 | 보상상 비용 | 채점상 비용 |
| --- | --- | --- |
| progress 0.3 | -8 | 에피소드 0점 |
| progress 0.9 | **-2** | 에피소드 0점 |

채점은 언제 부딪히든 전부를 뺏는데 보상은 **늦게 부딪힐수록 싸게** 만든다.
정책 입장의 최적 전략은 끝까지 밀어붙이는 것이고, 실제로 그렇게 학습됐다
(progress 0.896 / 충돌 20.6%).

### 실험 기록

| run | 바꾼 것 | 공식 채점 |
| --- | --- | --- |
| (기준) | 기본 보상 | **0.63281** |
| `..._v1` | overlap/offroad -10, below_ttc -0.5, comfort 0.2, `final_activation=null`, tau 0.01, alpha 0.05 | 0.479 (스윕) |
| `..._v2` | v1에서 dense 항 제거 | 0.510 (스윕) |
| `..._v3` | **overlap -3 하나만** (구조·tau·alpha는 기준값 유지) | 0.62306 |
| `..._v4` | v3 + `comfort 0.03` | ← 지금 |

**v1의 오류**: `comfort`와 `below_ttc`는 **매 스텝** 누적된다. 0.2 / -0.5면 에피소드
총합이 +10.4 / -10이 되어 진행 보상(+9)을 뒤집는다.

**v2가 알려준 것**: dense 항을 빼도 결과는 같았다. 두 run 모두 `model_320`에서
0.65 -> 0.37로 무너졌는데, 섹션 11 파인튜닝은 같은 지점에서 0.578을 유지했다.
붕괴의 원인은 보상이 아니라 v1/v2에만 있던 것들 - 특히 `final_activation=null`.
relu를 끄면 warm start로 받은 크리틱이 훈련 때와 다른 값을 뱉기 시작하고,
actor는 그 크리틱을 타고 올라가므로 첫 이터레이션부터 끌려간다.

**v3가 알려준 것**: 구조를 그대로 두고 `overlap`만 -1 -> -3으로 올리자 처음으로
의도한 효과가 나왔다.

| 지표 | 기준 | v3 |
| --- | --- | --- |
| overlap | 0.20601 | **0.17167** |
| comfort | 0.64694 | **0.56019** |
| progress_ratio | 0.89590 | 0.87632 |
| 총점 | 0.63281 | 0.62306 |

게이트 통과율이 0.771 -> 0.797로 올라 **+0.021**을 벌었지만, 품질 항이
0.821 -> 0.781로 떨어져 **-0.032**를 잃었다. 손실의 대부분이 comfort다.

당연한 결과다 - 충돌을 피하려면 급제동하고 급하게 꺾어야 하고, 그러면 저크와
가속도가 nuPlan 임계값을 넘는다. 그런데 `comfort`는 기본 설정에서 꺼져 있어
정책이 지킬 이유가 없었다. 배점의 30%가 무방비였던 셈이다.

### v4: 충돌 억제는 유지하고 comfort를 보호한다

`overlap=-3.0`은 효과가 증명됐으니 그대로 두고 `comfort`만 켠다. 0.03이면
에피소드 총합 +1.6으로 진행 보상의 18% - 목표를 뒤집지 않으면서 부드러움을
지키게 하기에 충분하다.

게이트 0.797을 유지한 채 comfort만 0.647로 되돌리면 **0.643**, 진행까지
회복하면 **0.655**로 기준(0.633)을 넘는다.

`comfort`는 `base_config.yaml`에 주석 처리돼 있어 존재하지 않는 키다. Hydra
struct 모드에서는 `+` 접두사로 **추가**해야 한다 (`overlap`은 기존 키라 불필요).

`save_freq`는 20으로 낮췄다 - 6,400 스텝마다 저장되므로 세션이 끊겨도 진행이
남는다. 학습에는 영향이 없는 값이다.


In [ ]:
import os

os.chdir("/content/as-fast-as-anyone/V-Max")

# 보상 재조정의 출발점: 대회 공식 채점기로 0.63281을 낸 체크포인트.
RW_BASE = f"runs/{os.environ['FT_RUN']}/model/model_96320.pkl"
assert os.path.isfile(RW_BASE), f"없음: {RW_BASE}"
os.environ["RW_BASE"] = RW_BASE

assert os.environ.get("MIXTURE"), "MIXTURE 미설정 - 섹션 11의 POOL_WEIGHTS 셀 먼저 실행"
print("warm start:", RW_BASE)
print("run name:", os.environ["RW_RUN"])


In [ ]:
!uv run python vmax/scripts/training/train.py \
  algorithm=sac network/encoder=lq \
  algorithm.network.policy.layer_sizes=[256,256] \
  "algorithm.pretrained_params_path=$RW_BASE" \
  reward_config.overlap=-3.0 \
  +reward_config.comfort=0.03 \
  algorithm.learning_rate=6e-5 \
  algorithm.alpha=0.1 \
  total_timesteps=100_000 \
  num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=50000 algorithm.learning_start=200 \
  waymo_dataset=true \
  "mixture_datasets=$MIXTURE" \
  name_run=$RW_RUN log_freq=20 save_freq=20


### 중단된 학습 이어하기

`sac_trainer.py`는 `runs/<name_run>/model/train_state_latest.pkl`이 있으면
`pretrained_params_path`보다 **먼저** 그것을 읽는다 (params + 옵티마이저 상태 +
env_steps 전부, `save_freq` 이터레이션마다 저장). 남은 스텝만 계산해서 돌므로
**위 학습 셀을 인자 하나 바꾸지 않고 다시 실행하는 것이 곧 이어학습이다.**

세션이 끊겼다면 `/content`가 비므로 그 전에 섹션 1·2 → 6 → 4·5 → 섹션 11의
POOL_WEIGHTS 셀을 다시 돌려야 한다. 특히 **섹션 6의 `runs` 심링크**가 빠지면
`train_state_latest.pkl`을 못 찾아 `pretrained_params_path`로 조용히 폴백해서
처음부터 학습한다.

**보상 인자 4개와 `value.final_activation`은 절대 바꾸지 말 것.** 이 값들은 학습
상태에 저장되지 않고 매 실행마다 config에서 재구성되므로, 다르게 주면 에러 없이
전반부와 후반부가 다른 목표로 학습된 잡종 모델이 된다.

재개되면 로그에 이 두 줄이 뜬다:

```
-> Resuming full training state from runs/.../train_state_latest.pkl ...
-> N/300000 steps already done, M iterations remaining
```

`-> Loading pretrained params from ...`가 대신 뜨면 재개가 아니라 처음부터 도는
것이니 멈추고 아래 점검 셀을 확인한다.


In [ ]:
# 이어학습 전 점검 - 전부 True여야 위 학습 셀이 재개된다.
import os

os.chdir("/content/as-fast-as-anyone/V-Max")
d = f"runs/{os.environ['RW_RUN']}/model"

print("runs 심링크:", os.path.islink("runs"))
print("run 디렉터리:", os.path.isdir(d))
print("train_state_latest:", os.path.isfile(f"{d}/train_state_latest.pkl"))
print("학습 데이터:", os.path.isdir("/content/data/shards/mixture_pools"))
print("MIXTURE:", bool(os.environ.get("MIXTURE")))

if os.path.isdir(d):
    print("체크포인트:", sorted(os.listdir(d)))


## 14. 보상 재조정 결과 확인

세 가지를 순서대로 본다. TensorBoard는 학습이 도는 동안 별도 셀에서 띄워도 된다.

1. **TensorBoard** - 학습이 살아 있는지, 리턴이 오르는지
2. **체크포인트 스윕** - 어느 체크포인트가 제일 잘 도는지 (loss가 아니라 실제 주행 점수)
3. **대회 공식 채점** - 최종 판단 기준. 섹션 13 이전 최고점은 **0.63281**이었다

보상을 바꿨으므로 **학습 로그의 리턴 값은 이전 run과 비교할 수 없다** (스케일이
다르다). 비교 가능한 것은 2번과 3번의 점수뿐이다.


In [ ]:
# 학습 중에 실행해도 된다 (별도 셀에서 띄워놓고 학습 셀을 돌리는 식).
# runs/ 는 Drive 심링크라 세션이 끊겨도 지난 run들의 곡선이 그대로 남아 있다.
%load_ext tensorboard
%tensorboard --logdir /content/as-fast-as-anyone/V-Max/runs


In [ ]:
# 섹션 9/11과 같은 평가셋($EVAL_PATH)이라 이전 run들의 점수와 나란히 놓을 수 있다.
import os

os.chdir("/content/as-fast-as-anyone/V-Max")
!uv run python scripts/evaluate_checkpoints.py \
  --name_run "$RW_RUN" \
  --path_dataset "$EVAL_PATH" \
  --waymo_dataset true --batch_size 4


In [ ]:
# 스윕에서 고른 체크포인트를 대회 제출 형식으로 포장.
# RW_CKPT를 위 스윕 출력에서 rideflux_aggregate_score가 가장 높은 파일명으로 바꾼다.
# 제출물은 policy만 싣기 때문에 value 쪽 설정이 어떻든 network.py는 그대로 맞는다.
import os
import shutil

os.chdir("/content/as-fast-as-anyone/V-Max")

RW_CKPT = "model_final.pkl"
SUB_DIR = "/content/as-fast-as-anyone/dxchallenge_planning_eval/submission_collision"

ckpt = f"runs/{os.environ['RW_RUN']}/model/{RW_CKPT}"
assert os.path.isfile(ckpt), f"없음: {ckpt}"
shutil.copytree(
    "/content/as-fast-as-anyone/dxchallenge_planning_eval/submission_hanam_run1",
    SUB_DIR, dirs_exist_ok=True,
)
!uv run python scripts/export_policy_weights.py "{ckpt}" "{SUB_DIR}/weights.pkl"


In [ ]:
# 대회 공식 채점. 세션을 새로 열었다면 섹션 12의 uv sync 셀과 roadgraph 패치 셀을
# 먼저 돌려야 한다 (섹션 2가 repo를 새로 clone하므로 패치가 날아간다).
import os

os.chdir("/content/as-fast-as-anyone/dxchallenge_planning_eval")
print("평가셋:", os.environ["EVAL_PATH"])

!CUDA_VISIBLE_DEVICES=0 uv run evaluate.py \
    --path_dataset "$EVAL_PATH" \
    --submission submission_collision \
    --time_limit 0
